In [1]:
import os
import pandas as pd

orig_df = pd.read_parquet("/home/cry_more/ongoing/LogRecall/artifacts/orig_df.parquet")
template_df = pd.read_parquet("/home/cry_more/ongoing/LogRecall/artifacts/template_df.parquet")

In [2]:
orig_df = orig_df[:1000]
template_df = template_df[:1000]

In [12]:
template_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   BlockId   1000 non-null   str  
 1   Template  1000 non-null   str  
dtypes: str(2)
memory usage: 1.9 MB


In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-small-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1006.66it/s]


In [34]:
import faiss
def fun(df):
    logs = df['Template'].tolist()
    embed = model.encode(
        logs,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    dimension = embed.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(embed)
    return index
        

In [36]:
index = fun(template_df)

Batches: 100%|██████████| 16/16 [00:14<00:00,  1.14it/s]


In [39]:
query_embedding = model.encode(
    "ffdsjkls",
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

query_embedding = query_embedding.reshape(1, -1)

scores, indices = index.search(query_embedding, k=2)